# 比賽火箭：RocketPy 6-DOF 飛行與穩定性 Demo

這份 notebook 直接讀取 Balloon Popping Challenge 的 scenario YAML，建立與比賽相同的 HybridMotor、氧化劑槽、火箭本體、鼻錐與尾翼；發射地點使用台灣屏東九鵬座標，預設使用執行當天台灣時間 02:00 的 GFS 預報風場。模擬維持馬達原始全推力，TVC 與 roll torque 為零，不加入閉迴路控制器；目的是隔離被動氣動穩定性、CG/CP 與 weathercocking 的影響。

**與比賽對齊：**火箭 YAML、S0 baseline 的 90° 發射姿態、0.01 m 虛擬導軌、RK45、time step，以及 S0/S1 各自 100/150 秒上限。**刻意保留的 demo 差異：**S0 使用無風 standard atmosphere；S1 agent 會依目標選擇發射角，並在固定 Ensemble 風場中每步控制 TVC。本 demo 改用當天 GFS 且維持 neutral controls，避免目標導引控制器掩蓋 CG/CP 的影響。

最常調整的變數集中在下一個 code cell：

- `BODY_CG_SHIFT_M`：正值把本體 CG 往鼻端移動。
- `FIN_POSITION_SHIFT_M`：正值把尾翼往鼻端移動，會直接影響 CP。
- `TARGET_INITIAL_STATIC_MARGIN_CAL`：最適合做比較；可直接指定點火時 static margin，例如 `0.5`、`1.0`、`2.0` cal。
- `TARGET_INITIAL_CG_CP_GAP_M`：也可用公尺指定點火時 `CG - CP`；兩種 target 請擇一使用。
- `FORECAST_DATE_TW`：指定 GFS 日期；`None` 代表執行當天，`FORECAST_TIME_TW = None` 時使用該日 02:00。
- `TRAJECTORY_VIEW_SIZE_M`：固定軌跡圖的 East × North × AGL 完整視野，預設 `250 × 250 × 200 m`。
- `ATTITUDE_PLOT_LIMITS`：固定 attitude 2D 圖的 X/Y 範圍；預設時間 `0～20 s`、角度 `-100～+100°`。

本 notebook 的定義沿用 RocketPy `tail_to_nose` 座標：**正的 `CG - CP` 與正的 static margin 代表靜態穩定**。static margin 的單位 `cal`（caliber）是一個火箭直徑。

> `TARGET_INITIAL_STATIC_MARGIN_CAL` 會以數值方式移動本體 CG，CP 與 inertia 維持原設定。較極端的 margin 適合做敏感度比較，不代表已完成可製造的配重設計。

所有輸出都放在同一個 `rocketpy_demo_outputs` 資料夾，檔名尾綴包含實際 static margin 與台灣時間，例如 `_2.00cal_20260806_014500`，方便上下切換比較。

參考：[RocketPy First Simulation](https://docs.rocketpy.org/en/latest/user/first_simulation.html)、[GFS forecast](https://docs.rocketpy.org/en/latest/user/environment/1-atm-models/forecast.html)、[Rocket class / static margin](https://docs.rocketpy.org/en/latest/reference/classes/Rocket.html)、[Flight plots](https://docs.rocketpy.org/en/latest/user/flight.html)。

In [ ]:
import json
import sys
from datetime import date, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from rocketpy import Flight
from rocketpy.simulation import FlightDataExporter


def find_repo_root():
    marker = Path("BalloonPoppingGymEnv/envs/scenario_parameters")
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError("Please run this notebook from inside the repository.")



REPO_ROOT = find_repo_root()
DEMO_DIR = REPO_ROOT / "doc/workshop_demos/00_rocketpy_competition_demo"
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

from rocketpy_demo_setup import TAIWAN_TZ, prepare_demo

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
print(f"Repository: {REPO_ROOT}")

## 1. 使用者控制區

第一次開啟請執行一次 `Run All`。之後通常只需修改這一格，再從本格執行 `Run All Below`。scenario 0 與 1 目前使用同一套火箭設計參數，但保留 selector 方便未來配置演進。GFS 需要網路連線；資料無法取得時會明確停止，不會悄悄退回無風的 standard atmosphere。

In [ ]:
# ---- Design / stability controls ----
SCENARIO_NUMBER = 0
BODY_CG_SHIFT_M = 0.0       # + toward nose, - toward tail
FIN_POSITION_SHIFT_M = 0.0  # + toward nose, - toward tail
TARGET_INITIAL_STATIC_MARGIN_CAL = None  # e.g. 0.5, 1.0, 2.0; easiest comparison
TARGET_INITIAL_CG_CP_GAP_M = None       # e.g. 0.44 m = 1 cal; use only one target

# ---- Pingtung launch site and live weather ----
LAUNCH_SITE_NAME = "Pingtung Jiupeng Launch Site"
LAUNCH_LATITUDE_DEG = 22.1749259
LAUNCH_LONGITUDE_DEG = 120.8922531
LAUNCH_ELEVATION_M = 20.0
WEATHER_MODEL = "GFS"  # GFS CG/CP demo; standard_atmosphere = S0-aligned no-wind control
FORECAST_DATE_TW = None  # None = execution date; or e.g. date(2026, 8, 6)
FORECAST_TIME_TW = None  # None = selected date at 02:00 Taiwan time; or datetime
WIND_PROFILE_TOP_AGL_M = 2000.0

# ---- Launch and solver controls ----
LAUNCH_INCLINATION_DEG = 90.0  # matches S0 baseline; S1 uses target-dependent launch angles
LAUNCH_HEADING_DEG = 0.0       # 0=N, 90=E
RAIL_LENGTH_M = 0.01           # matches the competition environment
MAX_SIMULATION_TIME_S = 150.0  # scenario YAML still caps S0 at 100 s, S1 at 150 s
TERMINATE_ON_APOGEE = False
TRAJECTORY_VIEW_SIZE_M = (250.0, 250.0, 200.0)  # East x North x AGL
ATTITUDE_PLOT_LIMITS = ((0.0, 20.0), (-100.0, 100.0))  # time (s), angle (deg)

# ---- Output controls ----
SAVE_PLOTS = True
EXPORT_KML = True
EXPORT_CSV = True
OUTPUT_DIR = (
    REPO_ROOT
    / "doc/workshop_demos/00_rocketpy_competition_demo/rocketpy_demo_outputs"
)

## 2. 建立模擬環境與比賽火箭

這一格通常不需要修改。它把上一格的設定交給同資料夾的 [`rocketpy_demo_setup.py`](./rocketpy_demo_setup.py)：該 module 會讀取 scenario YAML、建立 Environment／HybridMotor／Rocket，並回傳後續畫圖與模擬需要的物件。scenario YAML 仍是比賽參數的唯一真實來源。

In [ ]:
demo = prepare_demo(
    repo_root=REPO_ROOT,
    scenario_number=SCENARIO_NUMBER,
    body_cg_shift_m=BODY_CG_SHIFT_M,
    fin_position_shift_m=FIN_POSITION_SHIFT_M,
    target_initial_static_margin_cal=TARGET_INITIAL_STATIC_MARGIN_CAL,
    target_initial_cg_cp_gap_m=TARGET_INITIAL_CG_CP_GAP_M,
    launch_latitude_deg=LAUNCH_LATITUDE_DEG,
    launch_longitude_deg=LAUNCH_LONGITUDE_DEG,
    launch_elevation_m=LAUNCH_ELEVATION_M,
    weather_model=WEATHER_MODEL,
    forecast_date_tw=FORECAST_DATE_TW,
    forecast_time_tw=FORECAST_TIME_TW,
    wind_profile_top_agl_m=WIND_PROFILE_TOP_AGL_M,
    output_dir=OUTPUT_DIR,
)

scenario_path = demo.scenario_path
simulation_cfg = demo.simulation_cfg
rocket_cfg = demo.rocket_cfg
environment = demo.environment
rocket = demo.rocket
effective_body_cg_m = demo.effective_body_cg_m
run_started_tw = demo.run_started_tw
launch_time_tw = demo.launch_time_tw
initial_margin_for_tag = demo.initial_margin_for_tag
RUN_SUFFIX = demo.run_suffix
output_path = demo.output_path

print(f"Loaded rocket: {scenario_path.relative_to(REPO_ROOT)}")
print(f"Launch site:   {LAUNCH_SITE_NAME}")
print(f"Coordinates:   {environment.latitude:.7f}, {environment.longitude:.7f}")
print(f"Forecast time: {environment.local_date.isoformat()}")
print(f"Weather model: {environment.atmospheric_model_type}")
print(f"Output suffix: {RUN_SUFFIX}")

## 3. Pre-flight visual check：先確認火箭配置、CG 與 CP

這一步刻意放在 `Flight` 模擬之前。請先確認火箭本體、motor、tank、鼻錐和尾翼位置合理，並檢查藍色 **Center of Mass** 是否位於紅色 **Static Center of Pressure** 的鼻端方向。若配置不如預期，回到使用者控制區調整後，重新執行到這一格即可，不必先跑飛行模擬。

In [ ]:
ignition_cg_m = float(rocket.center_of_mass(0))
static_cp_m = float(rocket.cp_position(0))
ignition_gap_m = ignition_cg_m - static_cp_m
ignition_margin_cal = ignition_gap_m / (2 * rocket.radius)
print(f"Ignition CG:        {ignition_cg_m:8.4f} m")
print(f"Static CP (Mach 0): {static_cp_m:8.4f} m")
print(f"CG - CP:            {ignition_gap_m:8.4f} m")
print(f"Static margin:      {ignition_margin_cal:8.4f} cal")
print("Blue = CG, red = static CP")

# RocketPy's own component drawing, matching the official tutorial.
# A slightly smaller legend keeps long component labels inside saved PNGs.
with plt.rc_context({"legend.fontsize": 8, "savefig.bbox": "tight"}):
    rocket.draw()
    if SAVE_PLOTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        rocket.draw(filename=str(output_path("00_rocket_layout_cg_cp", ".png")))

## 4. 確認本次模擬使用的風場

預設 GFS 提供隨高度變化的東向（East）與北向（North）風速；若控制區選擇 `standard_atmosphere`，曲線會顯示為無風。右圖的 `Wind from direction` 採氣象慣例：0° 表示風從北方來、90° 表示從東方來。這張圖與飛行模擬共用同一個 `Environment`，並以同一個 static-margin/time-stamp 尾綴儲存。

> Weathercocking 是火箭朝**來風方向**偏轉，不是單純被吹向下風處。Static margin 過大通常會讓氣動回復力矩更強，因此比較不同 runs 時要一起看風場、ground track、attitude/path angle 與 angle of attack。

In [ ]:
wind_altitude_agl = np.linspace(0, WIND_PROFILE_TOP_AGL_M, 301)
wind_altitude_asl = wind_altitude_agl + environment.elevation
wind_east = np.array([environment.wind_velocity_x(h) for h in wind_altitude_asl])
wind_north = np.array([environment.wind_velocity_y(h) for h in wind_altitude_asl])
wind_speed = np.hypot(wind_east, wind_north)
wind_from_direction = np.array(
    [environment.wind_direction(h) for h in wind_altitude_asl]
)
wind_direction_for_plot = wind_from_direction.astype(float).copy()
direction_wraps = np.flatnonzero(np.abs(np.diff(wind_direction_for_plot)) > 180) + 1
wind_direction_for_plot[direction_wraps] = np.nan

for altitude_agl in (10.0, 100.0, 500.0, 1000.0):
    altitude_asl = environment.elevation + altitude_agl
    print(
        f"{altitude_agl:6.0f} m AGL: "
        f"wind {float(environment.wind_speed(altitude_asl)):5.2f} m/s, "
        f"from {float(environment.wind_direction(altitude_asl)):6.1f} deg true"
    )

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.5), sharey=True)
axes[0].plot(wind_speed, wind_altitude_agl, lw=2.2, label="Wind speed")
axes[0].plot(wind_east, wind_altitude_agl, label="East component")
axes[0].plot(wind_north, wind_altitude_agl, label="North component")
axes[0].axvline(0, color="0.4", lw=0.8)
axes[0].set(
    title="Wind speed and components",
    xlabel="Wind velocity (m/s)",
    ylabel="Altitude AGL (m)",
)
axes[0].legend()
axes[1].plot(wind_direction_for_plot, wind_altitude_agl, color="tab:purple", lw=2)
axes[1].set(
    title="Wind direction (meteorological)",
    xlabel="Wind from direction (deg true)",
)
axes[1].set_xlim(0, 360)
axes[1].set_xticks([0, 90, 180, 270, 360])
fig.suptitle(
    f"{LAUNCH_SITE_NAME} | {environment.local_date:%Y-%m-%d %H:%M %Z}"
)
fig.tight_layout()
if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path("01_wind_profile", ".png"), bbox_inches="tight")
plt.show()

## 5. 在模擬前檢查 CG、CP 與 static margin 隨時間的變化

這張圖最適合調參：左圖顯示燃燒期間的 CG 與 Mach 0 CP；右圖顯示兩者距離除以火箭直徑後的 static margin。若點火時 margin 為負，RocketPy 也會在建立 Flight 時發出不穩定警告。

In [ ]:
burnout_time = float(rocket.motor.burn_out_time)
initial_mass_kg = float(rocket.total_mass(0))
average_thrust_n = float(rocket.motor.average_thrust)
launch_gravity_m_s2 = float(environment.gravity(environment.elevation))
initial_thrust_to_weight = (
    average_thrust_n / (initial_mass_kg * launch_gravity_m_s2)
)
stability_time = np.linspace(0, burnout_time, 301)
center_of_mass = np.array([rocket.center_of_mass(t) for t in stability_time])
center_of_pressure = np.full_like(stability_time, float(rocket.cp_position(0)))
cg_cp_gap = center_of_mass - center_of_pressure
static_margin = cg_cp_gap / (2 * rocket.radius)

summary = {
    "effective body CG (m)": effective_body_cg_m,
    "CP at Mach 0 (m)": center_of_pressure[0],
    "initial total CG (m)": center_of_mass[0],
    "initial CG-CP gap (m)": cg_cp_gap[0],
    "initial static margin (cal)": static_margin[0],
    "burnout static margin (cal)": static_margin[-1],
    "initial total mass (kg)": initial_mass_kg,
    "average thrust (N)": average_thrust_n,
    "initial thrust-to-weight": initial_thrust_to_weight,
    "motor burn time (s)": burnout_time,
}
for label, value in summary.items():
    print(f"{label:32s}: {value:10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(stability_time, center_of_mass, label="CG (loaded rocket)", lw=2)
axes[0].plot(stability_time, center_of_pressure, label="CP (Mach 0)", lw=2)
axes[0].fill_between(
    stability_time, center_of_pressure, center_of_mass, alpha=0.16, label="CG - CP"
)
axes[0].set(title="CG and CP during burn", xlabel="Time (s)", ylabel="Axial position (m)")
axes[0].legend()
axes[1].plot(stability_time, static_margin, color="tab:green", lw=2)
axes[1].axhline(0, color="black", lw=1)
axes[1].fill_between(
    stability_time, 0, static_margin, where=static_margin >= 0, color="tab:green", alpha=0.14
)
axes[1].fill_between(
    stability_time, 0, static_margin, where=static_margin < 0, color="tab:red", alpha=0.20
)
axes[1].set(title="Static margin", xlabel="Time (s)", ylabel="Static margin (cal)")
fig.tight_layout()
if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path("02_stability", ".png"), bbox_inches="tight")
plt.show()

## 6. 執行 6-DOF 飛行模擬

Solver 與比賽環境一致使用 RK45、`time_overshoot=False`，最大 time step 直接讀 scenario 的 `simulation.time_step`。馬達維持 YAML 的 nominal full thrust，TVC 與 roll torque 固定為零；不加入比賽 agent 的閉迴路命令，讓穩定性差異更容易判讀。

> S0 baseline 以 90° 垂直姿態在 standard atmosphere 發射；S1 agent 則依目標選擇發射角，並在有風環境中每步更新 TVC。這份 demo 對齊 S0 的 90°，但刻意保留 neutral controls 與當天 GFS 風場，以隔離 CG/CP 對 weathercocking 的影響。預設火箭約 90.9 kg、平均推力 1080 N、燃燒 30 秒；在無風垂直控制實測可飛約 58 秒並達約 1.59 km AGL，因此低空提前落地不是燃料不足。只增加推進劑而維持相同推力反而會降低推重比。

In [ ]:
flight = Flight(
    rocket=rocket,
    environment=environment,
    rail_length=RAIL_LENGTH_M,
    inclination=LAUNCH_INCLINATION_DEG,
    heading=LAUNCH_HEADING_DEG,
    terminate_on_apogee=TERMINATE_ON_APOGEE,
    max_time=min(MAX_SIMULATION_TIME_S, simulation_cfg["max_time"]),
    max_time_step=simulation_cfg["time_step"],
    time_overshoot=False,
    verbose=False,
    ode_solver="RK45",
    name="Balloon Popping Challenge rocket (neutral controls)",
)
print(f"Flight simulated to t = {flight.t_final:.2f} s")
print(
    f"Apogee = {flight.apogee:.2f} m ASL "
    f"({flight.apogee - environment.elevation:.2f} m AGL) "
    f"at t = {flight.apogee_time:.2f} s"
)
print(f"Maximum speed = {flight.max_speed:.2f} m/s")
print(f"Maximum dynamic pressure = {flight.max_dynamic_pressure / 1000:.2f} kPa")
print(f"Rail-exit speed = {flight.out_of_rail_velocity:.2f} m/s")
if flight.t_final < burnout_time:
    print(
        f"Simulation ended {burnout_time - flight.t_final:.2f} s before "
        "motor burnout; adding propellant at the same thrust would lower T/W."
    )

## 7. 飛行軌跡、Attitude Angle 與 weathercocking 指標

除了 3D 軌跡與地面投影，下面也畫出：

1. `Flight Path Angle` 與 `Attitude Angle`：穩定火箭的兩條線通常應接近。
2. AGL 高度與速度。
3. Mach、dynamic pressure（Max-Q）與 angle of attack；這些可協助辨識氣動負載與失穩時段。
4. 每張圖標題都包含實際 initial static margin；配合相同尾綴的 GFS 風場圖，可比較 margin 過大時的 weathercocking。

3D 軌跡與 2D ground track 都使用 `TRAJECTORY_VIEW_SIZE_M` 的固定 East × North × AGL 視野，不會因短軌跡而自動放大。East 與 North 都以發射點 `(0, 0)` 置中；預設完整寬度為 250 m，所以水平軸端點會明確標示為 −125 與 +125 m。AGL 從地面 0 m 開始。`05_performance` 的高度軸使用相同 AGL 上限，並共用 `ATTITUDE_PLOT_LIMITS` 的時間 X 軸；其他物理量的 Y 軸各自保留合適尺度。

In [ ]:
def save_figure(fig, stem):
    if SAVE_PLOTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path(stem, ".png"), bbox_inches="tight")


run_label = f"Initial static margin = {initial_margin_for_tag:.2f} cal"
reference_wind_altitude_asl = environment.elevation + 100.0
reference_wind_speed = float(environment.wind_speed(reference_wind_altitude_asl))
reference_wind_from = float(
    environment.wind_direction(reference_wind_altitude_asl)
)
view_east_m, view_north_m, view_altitude_m = map(
    float, TRAJECTORY_VIEW_SIZE_M
)
if min(view_east_m, view_north_m, view_altitude_m) <= 0:
    raise ValueError("TRAJECTORY_VIEW_SIZE_M values must all be positive.")
east_limits_m = (-view_east_m / 2, view_east_m / 2)
north_limits_m = (-view_north_m / 2, view_north_m / 2)
altitude_limits_m = (0.0, view_altitude_m)
east_ticks_m = np.linspace(*east_limits_m, 5)
north_ticks_m = np.linspace(*north_limits_m, 5)
fig = plt.figure(figsize=(13, 5.5))
ax_3d = fig.add_subplot(1, 2, 1, projection="3d")
ax_3d.plot(flight.x[:, 1], flight.y[:, 1], flight.altitude[:, 1], lw=2)
ax_3d.scatter(flight.x[0, 1], flight.y[0, 1], flight.altitude[0, 1], color="black", label="Launch")
ax_3d.scatter(flight.x[-1, 1], flight.y[-1, 1], flight.altitude[-1, 1], color="red", marker="x", label="End")
ax_3d.set(
    title=f"3D flight trajectory | {run_label}",
    xlabel="East (m)",
    ylabel="North (m)",
    zlabel="Altitude AGL (m)",
    xlim=east_limits_m,
    ylim=north_limits_m,
    zlim=altitude_limits_m,
)
ax_3d.set_xticks(east_ticks_m)
ax_3d.set_yticks(north_ticks_m)
ax_3d.set_box_aspect((view_east_m, view_north_m, view_altitude_m))
ax_3d.legend()
ax_ground = fig.add_subplot(1, 2, 2)
ground_color = ax_ground.scatter(
    flight.x[:, 1], flight.y[:, 1], c=flight.time, s=8, cmap="viridis"
)
ax_ground.scatter(0, 0, color="black", marker="^", label="Launch")
ax_ground.set_aspect("equal", adjustable="box")
ax_ground.set(
    title=(
        f"Ground track | wind at 100 m AGL: {reference_wind_speed:.1f} m/s "
        f"from {reference_wind_from:.0f} deg"
    ),
    xlabel="East (m)",
    ylabel="North (m)",
    xlim=east_limits_m,
    ylim=north_limits_m,
)
ax_ground.set_xticks(east_ticks_m)
ax_ground.set_yticks(north_ticks_m)
ax_ground.legend()
fig.colorbar(ground_color, ax=ax_ground, label="Time (s)")
fig.suptitle(
    f"Fixed view: {view_east_m:g} x {view_north_m:g} x "
    f"{view_altitude_m:g} m (East x North x AGL)"
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(fig, "03_trajectory")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.8))
attitude_time_limits_s, attitude_angle_limits_deg = ATTITUDE_PLOT_LIMITS
if attitude_time_limits_s[1] <= attitude_time_limits_s[0] or attitude_angle_limits_deg[1] <= attitude_angle_limits_deg[0]:
    raise ValueError("ATTITUDE_PLOT_LIMITS must contain increasing (min, max) pairs.")
ax.plot(flight.path_angle[:, 0], flight.path_angle[:, 1], label="Flight path angle", lw=2)
ax.plot(flight.attitude_angle[:, 0], flight.attitude_angle[:, 1], label="Attitude angle", lw=2)
ax.plot(
    flight.lateral_attitude_angle[:, 0],
    flight.lateral_attitude_angle[:, 1],
    label="Lateral attitude angle",
    alpha=0.8,
)
ax.axvline(flight.apogee_time, color="0.4", ls=":", label="Apogee")
if burnout_time <= flight.t_final:
    ax.axvline(burnout_time, color="tab:red", ls="--", label="Burnout")
ax.set(
    title=f"Flight path and attitude angles | {run_label}",
    xlabel="Time (s)",
    ylabel="Angle (deg)",
    xlim=attitude_time_limits_s,
    ylim=attitude_angle_limits_deg,
)
ax.legend(ncols=2)
fig.tight_layout()
save_figure(fig, "04_attitude_angles")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for axis in axes.flat:
    axis.set_xlim(attitude_time_limits_s)
axes[0, 0].plot(flight.altitude[:, 0], flight.altitude[:, 1], color="tab:blue")
axes[0, 0].set(
    title="Altitude", ylabel="AGL (m)", ylim=altitude_limits_m
)
axes[0, 1].plot(flight.speed[:, 0], flight.speed[:, 1], color="tab:orange")
axes[0, 1].set(title="Speed", ylabel="m/s")
axes[1, 0].plot(flight.mach_number[:, 0], flight.mach_number[:, 1], color="tab:purple")
axes[1, 0].set(title="Mach number", xlabel="Time (s)", ylabel="Mach")
axes[1, 1].plot(
    flight.dynamic_pressure[:, 0],
    flight.dynamic_pressure[:, 1] / 1000,
    color="tab:red",
    label="Dynamic pressure",
)
axes[1, 1].set(title="Aerodynamic loading", xlabel="Time (s)", ylabel="q (kPa)")
aoa_axis = axes[1, 1].twinx()
aoa_axis.plot(
    flight.angle_of_attack[:, 0],
    flight.angle_of_attack[:, 1],
    color="tab:green",
    alpha=0.8,
    label="Angle of attack",
)
aoa_axis.set_ylabel("Angle of attack (deg)")
for axis in axes.flat:
    axis.axvline(flight.apogee_time, color="0.4", ls=":", lw=1)
    if burnout_time <= flight.t_final:
        axis.axvline(burnout_time, color="tab:red", ls="--", lw=1)
handles_1, labels_1 = axes[1, 1].get_legend_handles_labels()
handles_2, labels_2 = aoa_axis.get_legend_handles_labels()
axes[1, 1].legend(handles_1 + handles_2, labels_1 + labels_2, loc="best")
fig.suptitle(f"Flight performance dashboard | {run_label}")
fig.tight_layout()
save_figure(fig, "05_performance")
plt.show()

## 8. 匯出同資料夾、帶 static-margin/time-stamp 尾綴的結果

RocketPy 1.13 使用 `FlightDataExporter`；產生的 `.kml` 可直接在 Google Earth 開啟。所有 PNG、KML、CSV 與 metadata JSON 都放在同一個 `OUTPUT_DIR`，並共用同一個 `_實際值cal_時間戳` 尾綴。`altitude_mode="relativetoground"` 會把軌跡高度視為 launch site 地面以上高度，適合此 demo 的發射場局部平坦地面假設。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exporter = FlightDataExporter(flight, name="Competition rocket demo")
if EXPORT_KML:
    kml_path = output_path("competition_rocket_trajectory", ".kml")
    exporter.export_kml(
        file_name=str(kml_path),
        time_step=0.1,
        extrude=True,
        altitude_mode="relativetoground",
    )
    print(f"Google Earth KML: {kml_path}")
if EXPORT_CSV:
    csv_path = output_path("competition_rocket_flight", ".csv")
    exporter.export_data(
        str(csv_path),
        "x",
        "y",
        "z",
        "speed",
        "attitude_angle",
        "angle_of_attack",
        "mach_number",
        "dynamic_pressure",
        time_step=0.05,
    )
    print(f"Flight CSV:       {csv_path}")

metadata = {
    "run_timestamp_taipei": run_started_tw.isoformat(),
    "forecast_time_taipei": environment.local_date.isoformat(),
    "forecast_time_utc": environment.datetime_date.isoformat(),
    "weather_model": environment.atmospheric_model_type,
    "launch_site": LAUNCH_SITE_NAME,
    "latitude_deg": float(environment.latitude),
    "longitude_deg": float(environment.longitude),
    "elevation_m": float(environment.elevation),
    "launch_inclination_deg": float(LAUNCH_INCLINATION_DEG),
    "launch_heading_deg_true": float(LAUNCH_HEADING_DEG),
    "rail_length_m": float(RAIL_LENGTH_M),
    "control_mode": "full nominal thrust; zero TVC and roll torque",
    "initial_static_margin_cal": float(initial_margin_for_tag),
    "initial_cg_cp_gap_m": float(rocket.center_of_mass(0) - rocket.cp_position(0)),
    "trajectory_view_size_m": [
        view_east_m, view_north_m, view_altitude_m
    ],
    "attitude_plot_limits": [
        list(attitude_time_limits_s),
        list(attitude_angle_limits_deg),
    ],
    "wind_10m_agl_m_s": float(environment.wind_speed(environment.elevation + 10)),
    "wind_100m_agl_m_s": reference_wind_speed,
    "wind_from_100m_agl_deg_true": reference_wind_from,
    "apogee_asl_m": float(flight.apogee),
    "apogee_time_s": float(flight.apogee_time),
    "maximum_speed_m_s": float(flight.max_speed),
    "maximum_dynamic_pressure_pa": float(flight.max_dynamic_pressure),
}
metadata_path = output_path("run_metadata", ".json")
metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Run metadata:      {metadata_path}")
print("\nFiles for this run:")
for path in sorted(OUTPUT_DIR.glob(f"*{RUN_SUFFIX}*")):
    print(f"  - {path.name}")

## 建議的小實驗

每次只改一個旋鈕並記錄 initial static margin、最大 angle of attack 與軌跡：

1. 依序設定 `TARGET_INITIAL_STATIC_MARGIN_CAL = 0.5, 1.0, 2.0, 3.0`，每次 Run All；輸出會留在同一資料夾且不互相覆蓋。
2. 固定 CG，讓 `FIN_POSITION_SHIFT_M` 從 `-0.2` 掃到 `+0.2` m，觀察 CP 與姿態。
3. 固定 90° 發射與 static margin，只切換 `GFS`／`standard_atmosphere`，分離風場與火箭設計的影響。
4. 比較同一 GFS 時刻下的 `01_wind_profile`、`03_trajectory`、`04_attitude_angles` 與 `05_performance`；注意高 static margin 的典型問題是更強的迎風轉向（weathercocking），不等同於單純往下風漂移。

> 這是設計與教學工具，不是飛行安全認證。實體火箭還需要納入結構、致動器、風場不確定性、感測器、控制器、製造公差與 Monte Carlo 驗證。